# Flight Price Prediction Model

### Import Libraries
This cell imports all necessary libraries for data manipulation, machine learning pipeline creation, and model evaluation.

In [55]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from warnings import filterwarnings
filterwarnings('ignore')

### Load Dataset
Loads the `Clean_Dataset.csv` file into a pandas DataFrame named `df`.

In [56]:
df = pd.read_csv('Clean_Dataset.csv')

### Display First 5 Rows
Shows the initial few rows of the DataFrame to get a glimpse of the data structure and content.

In [57]:
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


### Check DataFrame Shape
Prints the number of rows and columns in the DataFrame.

In [58]:
df.shape

(300153, 12)

### Display DataFrame Information
Provides a summary of the DataFrame, including column names, non-null counts, and data types, to identify any missing values or incorrect types.

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        300153 non-null  int64  
 1   airline           300153 non-null  object 
 2   flight            300153 non-null  object 
 3   source_city       300153 non-null  object 
 4   departure_time    300153 non-null  object 
 5   stops             300153 non-null  object 
 6   arrival_time      300153 non-null  object 
 7   destination_city  300153 non-null  object 
 8   class             300153 non-null  object 
 9   duration          300153 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), object(8)
memory usage: 27.5+ MB


### Check for Null Values
Calculates the total number of null (missing) values across the entire DataFrame.

In [60]:
df.isnull().sum().sum()

np.int64(0)

### Check for Duplicate Rows
Counts the number of duplicate rows in the DataFrame.

In [61]:
df.duplicated().sum()

np.int64(0)

### Display Descriptive Statistics
Generates descriptive statistics for numerical columns, including count, mean, standard deviation, min, max, and quartiles.

In [62]:
df.describe()

,Unnamed: 0,duration,days_left,price
count,300153.000000,300153.000000,300153.000000,300153.000000
mean,150076.000000,12.221021,26.004751,20889.660523
std,86646.852011,7.191997,13.561004,22697.767366
min,0.000000,0.830000,1.000000,1105.000000
25%,75038.000000,6.830000,15.000000,4783.000000
50%,150076.000000,11.250000,26.000000,7425.000000
75%,225114.000000,16.170000,38.000000,42521.000000
max,300152.000000,49.830000,49.000000,123071.000000


### Categorize Price into Bins
Creates a new categorical column `price_category` by dividing the `price` column into 'Cheap', 'Moderate', and 'Expensive' based on percentiles.

In [63]:
low = np.percentile(df['price'], 33)
high = np.percentile(df['price'], 66)

df['price_category'] = np.where(df['price'] <= low, 'Cheap',
                        np.where(df['price'] <= high, 'Moderate', 'Expensive'))

### Display DataFrame Head (with price_category)
Shows the DataFrame's first few rows again, now including the newly created `price_category`.

In [64]:
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,price_category
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,Moderate
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,Moderate
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,Moderate
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,Moderate
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,Moderate


### Analyze Average Price by Airline
Calculates and displays the mean price for each airline, sorted in ascending order, to understand pricing differences between carriers.

In [65]:
df.groupby('airline')['price'].mean().sort_values().astype(int)

,price
airline,
AirAsia,4091
Indigo,5324
GO_FIRST,5652
SpiceJet,6179
Air_India,23507
Vistara,30396


### One-Hot Encode Categorical Features
Applies one-hot encoding to specified categorical columns (`airline`, `source_city`, etc.) to convert them into a numerical format suitable for machine learning models.

In [66]:
columns = ['airline', 'source_city', 'destination_city', 'departure_time',
           'arrival_time', 'stops', 'class']

df = pd.get_dummies(df, columns=columns, drop_first=False)

### Display DataFrame Head (after encoding)
Shows the DataFrame's head after one-hot encoding, illustrating the new column structure.

In [67]:
df.head()

,Unnamed: 0,flight,duration,days_left,price,price_category,airline_AirAsia,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,airline_Vistara,source_city_Bangalore,source_city_Chennai,source_city_Delhi,source_city_Hyderabad,source_city_Kolkata,source_city_Mumbai,destination_city_Bangalore,destination_city_Chennai,destination_city_Delhi,destination_city_Hyderabad,destination_city_Kolkata,destination_city_Mumbai,departure_time_Afternoon,departure_time_Early_Morning,departure_time_Evening,departure_time_Late_Night,departure_time_Morning,departure_time_Night,arrival_time_Afternoon,arrival_time_Early_Morning,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night,stops_one,stops_two_or_more,stops_zero,class_Business,class_Economy
0,0,SG-8709,2.17,1,5953,Moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,True
1,1,SG-8157,2.33,1,5953,Moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True
2,2,I5-764,2.17,1,5956,Moderate,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True
3,3,UK-995,2.25,1,5955,Moderate,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,True
4,4,UK-963,2.33,1,5955,Moderate,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,True


### Drop Unnecessary Columns
Removes the `Unnamed: 0` (likely an index column) and `flight` columns as they are not needed for the model.

In [68]:
df.drop(columns=['Unnamed: 0', 'flight'], axis = 1, inplace = True)

### Display DataFrame Head (after dropping columns)
Shows the DataFrame's head after dropping the specified columns.

In [69]:
df.head()

,duration,days_left,price,price_category,airline_AirAsia,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,airline_Vistara,source_city_Bangalore,source_city_Chennai,source_city_Delhi,source_city_Hyderabad,source_city_Kolkata,source_city_Mumbai,destination_city_Bangalore,destination_city_Chennai,destination_city_Delhi,destination_city_Hyderabad,destination_city_Kolkata,destination_city_Mumbai,departure_time_Afternoon,departure_time_Early_Morning,departure_time_Evening,departure_time_Late_Night,departure_time_Morning,departure_time_Night,arrival_time_Afternoon,arrival_time_Early_Morning,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night,stops_one,stops_two_or_more,stops_zero,class_Business,class_Economy
0,2.17,1,5953,Moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,True
1,2.33,1,5953,Moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True
2,2.17,1,5956,Moderate,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True
3,2.25,1,5955,Moderate,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,True
4,2.33,1,5955,Moderate,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,True


### Define Features (X) and Target (y)
Separates the DataFrame into features (`X`, all columns except 'price' and 'price_category') and the target variable (`y`, which is 'price_category').

In [70]:
X = df.drop(columns=['price', 'price_category'], axis = 1)
y = df['price_category']

### Split Data into Training and Testing Sets
Divides the dataset into training (80%) and testing (20%) sets to evaluate the model's performance on unseen data, ensuring reproducibility with `random_state=42`.

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Build and Train the Random Forest Classifier Pipeline
Constructs a machine learning pipeline that includes imputation for missing values (if any), standard scaling of features, and a Random Forest Classifier. The pipeline is then trained on the training data.

In [72]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(random_state=42))])

### Evaluate Model Performance
Predicts on the test set and prints a classification report (precision, recall, f1-score, support) and the mean cross-validation accuracy to assess the model's generalization ability.

In [73]:
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
print("\nCross-validation scores:",
      cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy').mean())

              precision    recall  f1-score   support

       Cheap       0.94      0.96      0.95     19798
   Expensive       0.98      0.97      0.98     20433
    Moderate       0.93      0.93      0.93     19800

    accuracy                           0.95     60031
   macro avg       0.95      0.95      0.95     60031
weighted avg       0.95      0.95      0.95     60031


Cross-validation scores: 0.948313770149456


## Conclusion

The Random Forest Classifier pipeline demonstrates strong performance in categorizing flight prices. The classification report shows high precision, recall, and f1-scores across all three price categories (Cheap, Moderate, Expensive), with an overall accuracy of approximately 95%. The cross-validation score further confirms the model's robustness and generalization capability, yielding a similar average accuracy of around 94.8%. This indicates that the model is well-suited for predicting flight price categories based on the given features.